# CS453 - Two-Week Individual Project Workbook

**Name:** [AYA HAMAD SHALLOUF]  
**Student ID:** [09221024]  
**Project:** [Library Retrieval]


## 1. Task, PEAS and environment

**Identity and variant:** AYA HAMAD SHALLOUF, ID 09221024, Library Retrieval Agent.
My variant is 0: road A-D is removed, start is S and the target shelf zone is G.

**Chosen task:** A library retrieval agent starts at node S and must reach the
assigned shelf zone G, then retrieve one requested item. Retrieval is legal
only in the assigned zone.

**Inputs:** the library graph (nodes are locations, edges are unit-cost roads),
the start node S, the target shelf zone G, and the agent's item status.
**Output:** a sequence of legal actions (the route from S to G followed by the
retrieve action) and its total travel cost.
**Success criteria:** only legal actions are used, the agent reaches G, and the
item status becomes "held". For A*, the route must also have minimum cost.

### PEAS

| Component | Description |
|---|---|
| **Performance** | Item retrieved; only legal actions; minimum number of moves; few expanded nodes |
| **Environment** | Undirected graph of 8 locations (S, A-G) with unit-cost roads and road A-D removed |
| **Actuators** | Move to an adjacent location; retrieve the item |
| **Sensors** | Current location, neighbouring locations, target zone, item-possession status |

### Environment properties

- **Fully observable:** the whole map, the agent's location and its item status are known.
- **Deterministic:** an action always has the same result; no random failures.
- **Sequential:** each move changes the location and affects later choices and cost.
- **Static:** the map does not change while the agent plans or moves.
- **Discrete:** locations, actions and item status are countable.
- **Single agent:** no other agents act in this simulation.

### Three scope limits

1. The map is a small fixed graph with unit-cost roads; no weighted roads,
   closures or delays that appear during execution.
2. One agent and one requested item only; no multiple requests or other agents.
3. No physical details (shelf height, grasping, availability of the book);
   retrieval is a single symbolic action.

## 2. Representation

**State:** (location, has_item), for example ("S", False).
**Initial state:** ("S", False). The task is NOT complete at the start, because
the agent is not at G and does not hold the item.

**Legal actions:**
- MOVE to a neighbour of the current location in the variant graph.
- RETRIEVE: legal only if location == "G" and has_item is False.

**Result:** MOVE changes location and keeps has_item. RETRIEVE keeps the
location and sets has_item to True. Illegal actions are rejected with a message.

**Goal test:** has_item is True (which can only happen at G).

**Costs:** each move costs 1. RETRIEVE adds no travel cost (it is a service
action, not travel).

**Service legality:** retrieval anywhere except G is illegal.
**Repeated service:** once has_item is True, RETRIEVE is rejected
("item already held").


In [9]:
# Supplied data/helper: cite CS453 two-week project workbook if adapted.
from copy import deepcopy
import platform

REFERENCE_GRAPH = {
    "S": ["A", "B"],
    "A": ["S", "C", "D"],
    "B": ["S", "E"],
    "C": ["A", "F"],
    "D": ["A", "G"],
    "E": ["B", "G"],
    "F": ["C", "G"],
    "G": ["D", "E", "F"],
}
VARIANTS = [("A", "D", "G"), ("B", "E", "F"),
            ("C", "F", "E"), ("E", "G", "D")]

def make_variant(student_id):
    if not str(student_id).strip():
        raise ValueError("Enter your student ID first")
    variant = sum(ord(ch) for ch in str(student_id).strip()) % 4
    graph = deepcopy(REFERENCE_GRAPH)
    u, v, goal = VARIANTS[variant]
    graph[u].remove(v)
    graph[v].remove(u)
    return {"variant": variant, "graph": graph, "start": "S",
            "goal": goal, "removed_edge": (u, v)}

STUDENT_ID = "09221024"  # TODO: your student ID as text, preserving leading zeros.
if STUDENT_ID.strip():
    assigned = make_variant(STUDENT_ID)
    print(assigned)
else:
    print("Unfinished template: enter your ID to generate your variant.")
print("Python:", platform.python_version())


{'variant': 0, 'graph': {'S': ['A', 'B'], 'A': ['S', 'C'], 'B': ['S', 'E'], 'C': ['A', 'F'], 'D': ['G'], 'E': ['B', 'G'], 'F': ['C', 'G'], 'G': ['D', 'E', 'F']}, 'start': 'S', 'goal': 'G', 'removed_edge': ('A', 'D')}
Python: 3.13.18


### Supplied heuristic data for Task 6

These are shortest distances in the reference graph with all original roads open. Your assigned variant removes a road, so those relaxed-world distances cannot overestimate the remaining route cost. They are supplied task data; students implement the actual BFS and A* algorithms. For goal F: S=3, A=2, B=3, C=1, D=2, E=2, F=0, G=1. No extra hand-trace task is required.

In [10]:
# Supplied heuristic data: h is distance in the original graph before a road closure.
# Removing roads cannot make a route shorter, so these values remain lower bounds.
# h(goal) = 0. These tables apply to the original graph and its edge-deletion variants.
# If you add shortcuts/change costs, re-justify h or use h = 0 instead.
HEURISTICS = {'A': {'A': 0, 'B': 2, 'C': 1, 'D': 1, 'E': 3, 'F': 2, 'G': 2, 'S': 1}, 'B': {'A': 2, 'B': 0, 'C': 3, 'D': 3, 'E': 1, 'F': 3, 'G': 2, 'S': 1}, 'C': {'A': 1, 'B': 3, 'C': 0, 'D': 2, 'E': 3, 'F': 1, 'G': 2, 'S': 2}, 'D': {'A': 1, 'B': 3, 'C': 2, 'D': 0, 'E': 2, 'F': 2, 'G': 1, 'S': 2}, 'E': {'A': 3, 'B': 1, 'C': 3, 'D': 2, 'E': 0, 'F': 2, 'G': 1, 'S': 2}, 'F': {'A': 2, 'B': 3, 'C': 1, 'D': 2, 'E': 2, 'F': 0, 'G': 1, 'S': 3}, 'G': {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1, 'F': 1, 'G': 0, 'S': 3}, 'S': {'A': 1, 'B': 1, 'C': 2, 'D': 2, 'E': 2, 'F': 3, 'G': 3, 'S': 0}}
print("Heuristic tables available for targets:", ", ".join(sorted(HEURISTICS)))


Heuristic tables available for targets: A, B, C, D, E, F, G, S


## 4. Implement and test the simulator

Write functions for legal moves, state updates, service action and task completion. Reject invalid actions clearly. Do not mutate shared map data between tests. Search plans the route; the simulator verifies/exercises its moves and final service.


In [11]:
# Simulator for the Library Retrieval Agent (my own implementation).
# State = (location, has_item). Actions = ("MOVE", node) or ("RETRIEVE",).
# The map is only read, never modified, so tests cannot change shared data.

def initial_state(assigned):
    """Start location with the item not yet held."""
    return (assigned["start"], False)

def legal_actions(state, assigned):
    """All legal actions in a state."""
    location, has_item = state
    graph = assigned["graph"]
    if location not in graph:
        raise ValueError(f"Unknown location: {location}")
    actions = [("MOVE", neighbour) for neighbour in graph[location]]
    if location == assigned["goal"] and not has_item:
        actions.append(("RETRIEVE",))
    return actions

def explain_illegal(state, action, assigned):
    """Clear reason why an action is not allowed."""
    location, has_item = state
    if not action:
        return "empty action"
    if action[0] == "MOVE":
        if len(action) != 2:
            return "MOVE needs a destination"
        if action[1] not in assigned["graph"]:
            return f"{action[1]} is not a location on the map"
        return f"no road from {location} to {action[1]}"
    if action[0] == "RETRIEVE":
        if has_item:
            return "item already held (repeated retrieval)"
        return f"retrieval is only legal at {assigned['goal']}, not at {location}"
    return f"unknown action type {action[0]}"

def apply_action(state, action, assigned):
    """Return the new state, or raise ValueError for an illegal action."""
    if action not in legal_actions(state, assigned):
        raise ValueError(f"Illegal action {action} in state {state}: "
                         f"{explain_illegal(state, action, assigned)}")
    location, has_item = state
    if action[0] == "MOVE":
        return (action[1], has_item)
    return (location, True)  # RETRIEVE

def action_cost(action):
    """Moves cost 1; the service action adds no travel cost."""
    return 1 if action[0] == "MOVE" else 0

def is_goal(state):
    """Task is complete when the item is held."""
    return state[1]

def run_plan(actions, assigned):
    """Execute a list of actions from the start. Returns (final_state, cost)."""
    state = initial_state(assigned)
    cost = 0
    print("Start:", state)
    for action in actions:
        state = apply_action(state, action, assigned)
        cost += action_cost(action)
        print(f"  {action} -> {state}")
    return state, cost

In [12]:
from copy import deepcopy

graph_before = deepcopy(assigned["graph"])

# 1) A valid plan: S -> B -> E -> G, then retrieve
plan = [("MOVE", "B"), ("MOVE", "E"), ("MOVE", "G"), ("RETRIEVE",)]
final_state, cost = run_plan(plan, assigned)
print("Final:", final_state, "| cost:", cost, "| goal reached:", is_goal(final_state))

# 2) Invalid actions must be rejected with a clear message
invalid_tests = [
    ("jump S -> G (no road)",        initial_state(assigned), ("MOVE", "G")),
    ("retrieve at S (wrong place)",  initial_state(assigned), ("RETRIEVE",)),
    ("removed road A -> D",          ("A", False),            ("MOVE", "D")),
    ("retrieve twice",               ("G", True),             ("RETRIEVE",)),
    ("unknown location",             initial_state(assigned), ("MOVE", "Z")),
]
for name, state, action in invalid_tests:
    try:
        apply_action(state, action, assigned)
        print("NOT REJECTED:", name)
    except ValueError as error:
        print("Rejected ->", error)

# 3) The shared map must not have been changed
print("Map unchanged:", assigned["graph"] == graph_before)

Start: ('S', False)
  ('MOVE', 'B') -> ('B', False)
  ('MOVE', 'E') -> ('E', False)
  ('MOVE', 'G') -> ('G', False)
  ('RETRIEVE',) -> ('G', True)
Final: ('G', True) | cost: 3 | goal reached: True
Rejected -> Illegal action ('MOVE', 'G') in state ('S', False): no road from S to G
Rejected -> Illegal action ('RETRIEVE',) in state ('S', False): retrieval is only legal at G, not at S
Rejected -> Illegal action ('MOVE', 'D') in state ('A', False): no road from A to D
Rejected -> Illegal action ('RETRIEVE',) in state ('G', True): item already held (repeated retrieval)
Rejected -> Illegal action ('MOVE', 'Z') in state ('S', False): Z is not a location on the map
Map unchanged: True


## 5. Uninformed search: BFS

Implement BFS using a FIFO queue, repeated-state tracking and route reconstruction. Explain why it is uninformed and why it finds a minimum-move path on these unit-cost graphs. Show the route returned for your assigned mission and the agent executing its service action. Handle start=goal, no route and invalid locations correctly. No separate test-case table or results-analysis task is required.

In [13]:
# BFS (uninformed search) over states (location, has_item).
# It uses the simulator functions from Task 4 and never changes the map.
from collections import deque

def reconstruct_plan(parent, goal_state):
    """Follow parent links from the goal state back to the start."""
    actions = []
    state = goal_state
    while parent[state] is not None:
        state, action = parent[state]
        actions.append(action)
    actions.reverse()
    return actions

def bfs(assigned, start_state=None):
    """Return (plan, cost, expanded). plan is None when no route exists."""
    start = start_state if start_state is not None else initial_state(assigned)
    if start[0] not in assigned["graph"]:
        raise ValueError(f"Invalid start location: {start[0]}")
    if is_goal(start):                      # start is already complete
        return [], 0, 0

    frontier = deque([start])               # FIFO queue
    parent = {start: None}                  # also acts as the visited set
    expanded = 0

    while frontier:
        state = frontier.popleft()          # oldest state first
        expanded += 1
        for action in legal_actions(state, assigned):
            nxt = apply_action(state, action, assigned)
            if nxt in parent:               # repeated state: skip it
                continue
            parent[nxt] = (state, action)
            if is_goal(nxt):
                plan = reconstruct_plan(parent, nxt)
                return plan, sum(action_cost(a) for a in plan), expanded
            frontier.append(nxt)

    return None, None, expanded             # frontier empty: no route

In [14]:
from copy import deepcopy

# --- Assigned mission (variant 0: S -> G, road A-D removed) ---
plan, cost, expanded = bfs(assigned)
print("BFS plan:", plan)
print("Moves (cost):", cost, "| expanded states:", expanded)
print("\nSimulator executing the BFS plan:")
final_state, run_cost = run_plan(plan, assigned)
print("Final:", final_state, "| goal reached:", is_goal(final_state))

# --- Special cases (on copies, so the assigned map stays unchanged) ---
print("\nStart already complete:", bfs(assigned, ("G", True)))
print("Start at the target zone:", bfs(assigned, ("G", False)))

no_route = deepcopy(assigned)               # cut every road into G
for node in list(no_route["graph"]["G"]):
    no_route["graph"][node].remove("G")
no_route["graph"]["G"] = []
print("No route:", bfs(no_route))

try:
    bfs(assigned, ("Z", False))
except ValueError as error:
    print("Invalid location rejected ->", error)

BFS plan: [('MOVE', 'B'), ('MOVE', 'E'), ('MOVE', 'G'), ('RETRIEVE',)]
Moves (cost): 3 | expanded states: 7

Simulator executing the BFS plan:
Start: ('S', False)
  ('MOVE', 'B') -> ('B', False)
  ('MOVE', 'E') -> ('E', False)
  ('MOVE', 'G') -> ('G', False)
  ('RETRIEVE',) -> ('G', True)
Final: ('G', True) | goal reached: True

Start already complete: ([], 0, 0)
Start at the target zone: ([('RETRIEVE',)], 0, 1)
No route: (None, None, 6)
Invalid location rejected -> Invalid start location: Z


**BFS explanation:** BFS uses a FIFO queue, so it expands states in the order
they were found: first all states 1 action away, then 2, and so on. It is
uninformed because it uses no estimate of how close a state is to the goal
(no heuristic); it only uses the order of discovery. The parent dictionary
stores how each state was first reached, so repeated states are skipped and the
route can be rebuilt backwards from the goal. Since every move costs 1 (and
RETRIEVE costs 0 and appears once at the end), the first goal found uses the
fewest moves. If the queue becomes empty, there is no route.

## 6. Informed search: A*

Implement A* with f(n) = g(n) + h(n), using a priority queue, best-cost tracking and parent reconstruction. Use the supplied HEURISTICS table for your assigned goal. Explain what g and h mean, why the supplied h is admissible on the provided variant, and when it would need revising. Test the goal on removal from the frontier. Show the assigned route and execute the service action. A* and BFS should find the same minimum move count here; their paths may differ. No separate comparison table or limitations essay is required.

In [15]:
# A* (informed search) over states (location, has_item), f(n) = g(n) + h(n).
# It reuses the simulator (Task 4) and reconstruct_plan (Task 5).
import heapq
from itertools import count

def heuristic(state, assigned):
    """Supplied h for my target zone; 0 once the item is held."""
    location, has_item = state
    if has_item:
        return 0
    return HEURISTICS[assigned["goal"]][location]

def astar(assigned, start_state=None):
    """Return (plan, cost, expanded). plan is None when no route exists."""
    start = start_state if start_state is not None else initial_state(assigned)
    if start[0] not in assigned["graph"]:
        raise ValueError(f"Invalid start location: {start[0]}")

    tie = count()                            # breaks ties in f in a fixed order
    frontier = [(heuristic(start, assigned), 0, next(tie), start)]  # priority queue
    best_g = {start: 0}                      # cheapest known cost to each state
    parent = {start: None}
    expanded = 0

    while frontier:
        f, g, _, state = heapq.heappop(frontier)   # smallest f first
        if g > best_g[state]:                # outdated entry, a cheaper one exists
            continue
        if is_goal(state):                   # goal test on REMOVAL from frontier
            plan = reconstruct_plan(parent, state)
            return plan, g, expanded
        expanded += 1
        for action in legal_actions(state, assigned):
            nxt = apply_action(state, action, assigned)
            g2 = g + action_cost(action)
            if nxt not in best_g or g2 < best_g[nxt]:   # new or cheaper route
                best_g[nxt] = g2
                parent[nxt] = (state, action)
                heapq.heappush(frontier,
                               (g2 + heuristic(nxt, assigned), g2, next(tie), nxt))

    return None, None, expanded              # frontier empty: no route

In [16]:
from copy import deepcopy

# --- Assigned mission (variant 0: S -> G, road A-D removed) ---
plan, cost, expanded = astar(assigned)
print("A* plan:", plan)
print("Moves (cost):", cost, "| expanded states:", expanded)
print("\nSimulator executing the A* plan:")
final_state, run_cost = run_plan(plan, assigned)
print("Final:", final_state, "| goal reached:", is_goal(final_state))

bfs_plan, bfs_cost, bfs_expanded = bfs(assigned)
print("\nBFS cost:", bfs_cost, "| A* cost:", cost, "| same minimum:", bfs_cost == cost)
print("Expanded states -> BFS:", bfs_expanded, "| A*:", expanded)

# --- Special cases (on copies, so the assigned map stays unchanged) ---
print("\nStart already complete:", astar(assigned, ("G", True)))
print("Start at the target zone:", astar(assigned, ("G", False)))

no_route = deepcopy(assigned)               # cut every road into G
for node in list(no_route["graph"]["G"]):
    no_route["graph"][node].remove("G")
no_route["graph"]["G"] = []
print("No route:", astar(no_route))

try:
    astar(assigned, ("Z", False))
except ValueError as error:
    print("Invalid location rejected ->", error)

# --- Is the supplied h admissible on MY variant? (h <= true remaining cost) ---
print("\nnode | h | true cost | h <= true")
for node in sorted(assigned["graph"]):
    h = heuristic((node, False), assigned)
    true_cost = bfs(assigned, (node, False))[1]
    print(f"  {node}  | {h} |     {true_cost}     | {h <= true_cost}")

A* plan: [('MOVE', 'B'), ('MOVE', 'E'), ('MOVE', 'G'), ('RETRIEVE',)]
Moves (cost): 3 | expanded states: 5

Simulator executing the A* plan:
Start: ('S', False)
  ('MOVE', 'B') -> ('B', False)
  ('MOVE', 'E') -> ('E', False)
  ('MOVE', 'G') -> ('G', False)
  ('RETRIEVE',) -> ('G', True)
Final: ('G', True) | goal reached: True

BFS cost: 3 | A* cost: 3 | same minimum: True
Expanded states -> BFS: 7 | A*: 5

Start already complete: ([], 0, 0)
Start at the target zone: ([('RETRIEVE',)], 0, 1)
No route: (None, None, 6)
Invalid location rejected -> Invalid start location: Z

node | h | true cost | h <= true
  A  | 2 |     3     | True
  B  | 2 |     2     | True
  C  | 2 |     2     | True
  D  | 1 |     1     | True
  E  | 1 |     1     | True
  F  | 1 |     1     | True
  G  | 0 |     0     | True
  S  | 3 |     3     | True


**A\* explanation:** A* uses a priority queue ordered by f(n) = g(n) + h(n).
g(n) is the real cost already paid from the start to state n. h(n) is the
supplied estimate of the remaining cost to my target zone G (from
HEURISTICS["G"], and 0 once the item is held). A* always takes the state with
the smallest f first. best_g stores the cheapest cost found for each state,
and a state is added again only if a cheaper route is found. parent is used
to rebuild the route. The goal is tested when a state is removed from the
queue, because only then we are sure no cheaper route to it is waiting.

**Why h is admissible:** the supplied h values are shortest distances in the
original graph, where all roads are open. My variant removes road A-D, and
removing a road can never make a route shorter, so h never overestimates the
real cost. For example, h(A) = 2, but the real cost from A is now 3. My check
table showed h <= real cost for every node.

**When h needs revising:** if I add shortcuts or change road costs, the old
values could be too high. Then I would recompute h or use h = 0.

**My result:** both algorithms found 3 moves (S -> B -> E -> G, then RETRIEVE).
BFS expanded 7 states and A* expanded 5, because h guides A* toward G.

## 10. Sources, assistance and questions

**Sources:**
I used the reference notebooks `graph-search` and `informed-search` for the ideas
behind BFS and A* (queue, parent dictionary, priority queue, and best-cost updates).
I wrote my own code and adapted it to the Library Retrieval Agent project and the
simulator I designed.

**AI assistance:**
I used Claude (Anthropic) to explain the assignment instructions and the concepts,
suggest and correct code, and review the results.

**Verification:**
I ran all cells and checked the results myself: the route and cost (3 moves),
rejection of invalid actions, that h never exceeds the real cost, and that BFS
and A* return the same cost.

## GitHub + Colab workflow

1. On the public starter, use Use this template > Create a new repository, choose Private, then invite the professor using the announced GitHub username.
2. Your copy already includes project.ipynb. Open it in Colab using authorized GitHub access or download/upload.
3. Save each work session explicitly to GitHub: File > Save a copy in/to GitHub, correct repo/branch and project.ipynb. Drive autosave alone does not update GitHub.
4. If integration fails, download the notebook from Colab, upload it to GitHub to replace project.ipynb, and commit. No terminal or embedded tokens are needed.
5. Commit real progress throughout the project. Complete README.md using the supplied template.
6. Reopen the committed notebook in a fresh Colab session and verify it runs. Submit repo URL + final commit URL in Classroom and Turn in.

See the student guide pages 5-6 and docs/github-colab-guide.md for details. The professor must accept the invitation before viewing private work; do not wait for acceptance to begin.


## 11. Submission checklist

- [x] Identity, chosen project and generated variant recorded
- [x] Task 1: PEAS and environment explained
- [x] Task 2: state, legal actions, transition, goal and costs defined
- [x] Task 4: legal movement, service and task goal implemented
- [x] Task 5: working BFS and assigned mission output
- [x] Task 6: working A*, admissible heuristic explanation and assigned mission output
- [x] Sources and assistance disclosed
- [x] README with task/variant/files/dependencies/run steps
- [x] Clean restart/run-all checked and saved
- [x] project.ipynb and README.md saved in my private GitHub repository
- [x] Professor invited; repository URL and final commit URL submitted in Classroom

Original task numbers are retained for reference. Tasks 3, 7, 8 and 9 are removed. There is no Day 7 checkpoint.